# 04-1. 전처리 후보 추출 및 수동 검토

이 노트북은 **원본 데이터를 수정하거나 삭제하지 않는다.** 문제 유형별 후보 기사를 `review_data/04-1_candidates`에 CSV로 저장하고, 사람이 `review_decision`, `drop_reason`, `memo`를 채워 04-2의 전처리 규칙을 결정하기 위한 단계다.

- `repeat_4gram`은 팀의 현재 방침대로 clean 데이터에서 제외할 후보로 표시한다.
- 나머지 규칙은 오류 확정이 아니라 **검토 후보**다. 특히 숫자·기자 표현·짧은 본문은 자동 삭제하지 않는다.
- test는 규칙을 새로 고르는 데 사용하지 않고, 이미 정한 규칙을 적용했을 때의 clean test 보조 평가용으로만 사용한다.

In [1]:
import os
import sys
from pathlib import Path

PROJECT = Path(r"C:\Users\KDS-30\Desktop\Projects\06_Team_Project4")
assert PROJECT.exists(), "프로젝트 경로를 확인하세요."
assert Path(sys.prefix).resolve() == (PROJECT / ".venv").resolve(), (
    "커널을 Python 3.11 (Team4 News GPU)로 바꾸세요."
)

os.environ["HF_HOME"] = str(PROJECT / ".cache" / "huggingface")
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"

DATASET_ID = "daekeun-ml/naver-news-summarization-ko"
SEED = 42
REVIEW_DIR = PROJECT / "review_data" / "04-1_candidates"
REVIEW_DIR.mkdir(parents=True, exist_ok=True)

print("결과 폴더:", REVIEW_DIR)

결과 폴더: C:\Users\KDS-30\Desktop\Projects\06_Team_Project4\review_data\04-1_candidates


In [2]:
import re

import numpy as np
import pandas as pd
from datasets import load_dataset
from huggingface_hub import HfApi

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 100)

api = HfApi()
dataset_revision = api.dataset_info(DATASET_ID).sha
raw_dataset = load_dataset(DATASET_ID, revision=dataset_revision)

frames = []
for split in ["train", "validation", "test"]:
    frame = raw_dataset[split].to_pandas().copy()
    frame.insert(0, "source_index", np.arange(len(frame), dtype=np.int64))
    frame.insert(0, "split", split)
    frame.insert(0, "row_id", split + ":" + frame["source_index"].astype(str))
    frames.append(frame)

all_df = pd.concat(frames, ignore_index=True)
assert all_df["row_id"].is_unique
print("dataset revision:", dataset_revision)
display(all_df.groupby("split", sort=False).size().rename("rows").to_frame())

dataset revision: 916dfe6aed651b2ecd9d04fa0a704f8953e90762


,rows
split,
train,22194
validation,2466
test,2740


In [3]:
# 검토용 특징을 계산합니다. 각 flag는 오류 판정이 아니라 후보 선별 기준입니다.
def words(text):
    return re.findall(r"[0-9A-Za-z가-힣]+", str(text).lower())

def has_repeat_4gram(text):
    tokens = words(text)
    grams = [tuple(tokens[i:i + 4]) for i in range(len(tokens) - 3)]
    return len(grams) != len(set(grams))

def overlap_ratio(document, summary):
    summary_set = set(words(summary))
    if not summary_set:
        return np.nan
    return len(summary_set & set(words(document))) / len(summary_set)

def normalized_numbers(text):
    # 20 30처럼 숫자 사이의 공백만 제거해 2030과 같은 표기를 같은 후보로 취급합니다.
    text = re.sub(r"(?<=\d)\s+(?=\d)", "", str(text))
    return set(re.findall(r"\d+(?:[.,]\d+)?", text.replace(",", "")))

all_df["document_words"] = all_df["document"].map(lambda x: len(words(x)))
all_df["summary_words"] = all_df["summary"].map(lambda x: len(words(x)))
all_df["document_chars"] = all_df["document"].str.len()
all_df["summary_chars"] = all_df["summary"].str.len()
all_df["char_compression_ratio"] = all_df["summary_chars"] / all_df["document_chars"]
all_df["summary_document_overlap"] = all_df.apply(
    lambda row: overlap_ratio(row["document"], row["summary"]), axis=1
)
all_df["repeat_4gram"] = all_df["summary"].map(has_repeat_4gram)
all_df["low_overlap"] = all_df["summary_document_overlap"] < 0.30
all_df["extreme_ratio"] = (all_df["char_compression_ratio"] < 0.02) | (
    all_df["char_compression_ratio"] > 0.80
)
all_df["summary_longer"] = all_df["summary_chars"] > all_df["document_chars"]
all_df["short_document"] = all_df["document_words"] < 30
all_df["has_url"] = all_df["document"].str.contains(r"https?://|www\.", case=False, regex=True)
all_df["reporter_expression"] = all_df["document"].str.contains(
    r"기자|특파원|인턴기자", regex=True
)
all_df["copyright_phrase"] = all_df["document"].str.contains(
    r"무단[전재복제]|재배포[ ]?금지|저작권자", regex=True
)

document_numbers = all_df["document"].map(normalized_numbers)
summary_numbers = all_df["summary"].map(normalized_numbers)
all_df["document_numbers"] = document_numbers.map(lambda x: ", ".join(sorted(x)))
all_df["summary_numbers"] = summary_numbers.map(lambda x: ", ".join(sorted(x)))
all_df["numbers_only_in_summary"] = [
    ", ".join(sorted(s - d)) for d, s in zip(document_numbers, summary_numbers)
]
all_df["number_mismatch_candidate"] = all_df["numbers_only_in_summary"].ne("")

display(all_df[["repeat_4gram", "low_overlap", "extreme_ratio", "summary_longer", "number_mismatch_candidate"]].sum().to_frame("candidates"))

,candidates
repeat_4gram,5433
low_overlap,12
extreme_ratio,2823
summary_longer,2132
number_mismatch_candidate,777


In [4]:
# 중복/제목 그룹을 계산합니다. 같은 제목은 후속 기사일 수 있으므로 자동 삭제 대상으로 삼지 않습니다.
summary_doc_counts = all_df.groupby("summary")["document"].nunique()
shared_summaries = summary_doc_counts[summary_doc_counts > 1].index
all_df["different_document_same_summary"] = all_df["summary"].isin(shared_summaries)

title_doc_counts = all_df.groupby("title")["document"].nunique()
reused_titles = title_doc_counts[title_doc_counts > 1].index
all_df["same_title_different_document"] = all_df["title"].isin(reused_titles)

all_df["duplicate_group_id"] = pd.NA
all_df.loc[all_df["different_document_same_summary"], "duplicate_group_id"] = (
    "shared_summary_" + all_df.loc[all_df["different_document_same_summary"]]
    .groupby("summary").ngroup().astype(str)
)
all_df["title_group_id"] = pd.NA
all_df.loc[all_df["same_title_different_document"], "title_group_id"] = (
    "reused_title_" + all_df.loc[all_df["same_title_different_document"]]
    .groupby("title").ngroup().astype(str)
)

print("서로 다른 본문-같은 요약 행 수:", int(all_df["different_document_same_summary"].sum()))
print("같은 제목-서로 다른 본문 행 수:", int(all_df["same_title_different_document"].sum()))

서로 다른 본문-같은 요약 행 수: 100
같은 제목-서로 다른 본문 행 수: 1960


In [5]:
# CSV마다 같은 검토 열을 둡니다. decision은 keep / drop / undecided 중 하나로 직접 작성합니다.
TEXT_COLUMNS = [
    "row_id", "split", "source_index", "date", "category", "press", "title", "link",
    "document", "summary", "document_words", "summary_words",
    "document_chars", "summary_chars", "char_compression_ratio",
    "summary_document_overlap", "document_numbers", "summary_numbers",
    "numbers_only_in_summary", "duplicate_group_id", "title_group_id",
]
REVIEW_COLUMNS = ["review_decision", "drop_reason", "memo"]

def export_candidates(name, mask, description, recommended_action="manual_review"):
    result = all_df.loc[mask, TEXT_COLUMNS].copy()
    result.insert(3, "candidate_type", name)
    result.insert(4, "candidate_description", description)
    result.insert(5, "recommended_action", recommended_action)
    for column in REVIEW_COLUMNS:
        result[column] = ""
    path = REVIEW_DIR / f"{name}.csv"
    result.to_csv(path, index=False, encoding="utf-8-sig")
    return {"file": path.name, "rows": len(result), "description": description}

exports = []
exports.append(export_candidates(
    "01_repeat_4gram", all_df["repeat_4gram"],
    "요약 안에서 동일한 4어절 묶음이 2회 이상 반복됨", "drop_clean_split"
))
exports.append(export_candidates("02_low_overlap", all_df["low_overlap"], "요약 어절의 본문 출현 비율이 30% 미만"))
exports.append(export_candidates("03_extreme_ratio", all_df["extreme_ratio"], "요약/본문 글자 비율이 2% 미만 또는 80% 초과"))
exports.append(export_candidates("04_summary_longer", all_df["summary_longer"], "요약 글자 수가 본문 글자 수보다 큼"))
exports.append(export_candidates("05_number_mismatch", all_df["number_mismatch_candidate"], "본문에 없는 숫자가 요약에 있음; 표기 차이 등 오탐 가능"))
exports.append(export_candidates("06_shared_summary", all_df["different_document_same_summary"], "서로 다른 본문이 같은 요약을 공유"))
exports.append(export_candidates("07_reused_title", all_df["same_title_different_document"], "같은 제목이 서로 다른 본문에 사용됨"))
exports.append(export_candidates("08_url", all_df["has_url"], "본문에 URL이 포함됨", "manual_review_or_remove_url_only"))
exports.append(export_candidates("09_copyright", all_df["copyright_phrase"], "본문에 저작권/재배포 문구가 포함됨", "manual_review_or_remove_phrase_only"))
exports.append(export_candidates("10_short_document", all_df["short_document"], "본문이 30어절 미만; 삭제가 아닌 품질 확인 후보"))

# 기자 표현은 많으므로 전수를 출력하지 않고, split별 고정 시드 표본 50건씩만 저장합니다.
reporter_sample = (all_df.loc[all_df["reporter_expression"]]
    .groupby("split", group_keys=False)
    .apply(lambda x: x.sample(min(len(x), 50), random_state=SEED))
)
reporter_mask = all_df["row_id"].isin(reporter_sample["row_id"])
exports.append(export_candidates("11_reporter_expression_sample", reporter_mask, "기자/특파원 표현 표본; 자동 삭제하지 않음"))

summary_df = pd.DataFrame(exports)
summary_df.to_csv(REVIEW_DIR / "00_export_summary.csv", index=False, encoding="utf-8-sig")
display(summary_df)

C:\Users\KDS-30\AppData\Local\Temp\ipykernel_29028\1103811710.py:40: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), 50), random_state=SEED))


,file,rows,description
0,01_repeat_4gram.csv,5433,요약 안에서 동일한 4어절 묶음이 2회 이상 반복됨
1,02_low_overlap.csv,12,요약 어절의 본문 출현 비율이 30% 미만
2,03_extreme_ratio.csv,2823,요약/본문 글자 비율이 2% 미만 또는 80% 초과
3,04_summary_longer.csv,2132,요약 글자 수가 본문 글자 수보다 큼
4,05_number_mismatch.csv,777,본문에 없는 숫자가 요약에 있음; 표기 차이 등 오탐 가능
5,06_shared_summary.csv,100,서로 다른 본문이 같은 요약을 공유
6,07_reused_title.csv,1960,같은 제목이 서로 다른 본문에 사용됨
7,08_url.csv,83,본문에 URL이 포함됨
8,09_copyright.csv,25,본문에 저작권/재배포 문구가 포함됨
9,10_short_document.csv,1505,본문이 30어절 미만; 삭제가 아닌 품질 확인 후보


In [6]:
# 한 기사에 여러 문제가 겹친 경우를 한 파일에서 비교할 수 있게 통합 목록도 저장합니다.
FLAG_COLUMNS = [
    "repeat_4gram", "low_overlap", "extreme_ratio", "summary_longer",
    "number_mismatch_candidate", "different_document_same_summary",
    "same_title_different_document", "has_url", "copyright_phrase", "short_document",
]
any_flag = all_df[FLAG_COLUMNS].any(axis=1)
overview = all_df.loc[any_flag, TEXT_COLUMNS + FLAG_COLUMNS].copy()
overview["matched_rules"] = overview[FLAG_COLUMNS].apply(
    lambda row: ", ".join(row.index[row].tolist()), axis=1
)
for column in REVIEW_COLUMNS:
    overview[column] = ""
overview.to_csv(REVIEW_DIR / "00_all_flagged_rows.csv", index=False, encoding="utf-8-sig")

run_info = pd.DataFrame([{
    "dataset_id": DATASET_ID,
    "dataset_revision": dataset_revision,
    "total_rows": len(all_df),
    "flagged_rows": len(overview),
    "note": "후보 파일이며 이 실행은 원본 데이터를 변경하지 않음",
}])
run_info.to_csv(REVIEW_DIR / "00_run_info.csv", index=False, encoding="utf-8-sig")

print(f"저장 완료: {REVIEW_DIR}")
print(f"여러 규칙에 걸린 통합 후보: {len(overview):,}건")
for path in sorted(REVIEW_DIR.glob("*.csv")):
    print("-", path.name)

저장 완료: C:\Users\KDS-30\Desktop\Projects\06_Team_Project4\review_data\04-1_candidates
여러 규칙에 걸린 통합 후보: 7,844건
- 00_all_flagged_rows.csv
- 00_export_summary.csv
- 00_run_info.csv
- 01_repeat_4gram.csv
- 02_low_overlap.csv
- 03_extreme_ratio.csv
- 04_summary_longer.csv
- 05_number_mismatch.csv
- 06_shared_summary.csv
- 07_reused_title.csv
- 08_url.csv
- 09_copyright.csv
- 10_short_document.csv
- 11_reporter_expression_sample.csv


## 검토 순서

1. `01_repeat_4gram.csv`: 현재 합의대로 clean split 제외 여부를 확인한다.
2. `02_low_overlap.csv`, `05_number_mismatch.csv`, `06_shared_summary.csv`를 우선 직접 비교해 실제 오답/짝 불일치만 `drop`으로 기록한다.
3. `03_extreme_ratio.csv`, `04_summary_longer.csv`, `10_short_document.csv`는 길이만으로 삭제하지 말고 표본을 본다.
4. URL·저작권은 행을 버리기보다 문구만 제거할지 판단한다. 기자 표현은 보존이 기본값이다.

`04-2`에서는 확정된 `review_decision=drop` 목록과 반복 4-gram 규칙만 읽어 clean 데이터셋을 만들고, 원본 split은 별도로 보존한다.